In [1]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from textblob import TextBlob
from nltk.sentiment import SentimentIntensityAnalyzer
import spacy
from collections import Counter
from wordcloud import WordCloud
from transformers import pipeline

In [2]:
file_path = '/content/final_combined_file.csv'
df = pd.read_csv(file_path)

In [3]:
print(df.head())
print(df.info())

                                           headlines  \
0  Nirmala Sitharaman to equal Morarji Desai’s re...   
1  ‘Will densify network, want to be at least no....   
2  Air India group to induct an aircraft every si...   
3  Red Sea woes: Exporters seek increased credit ...   
4  Air India group to induct a plane every 6 days...   

                                         description  \
0  With the presentation of the interim budget on...   
1  'In terms of market share, we aim to double it...   
2  Air India currently has 117 operational aircra...   
3  Rising attacks forced shippers to consider the...   
4  Apart from fleet expansion, 2024 will also see...   

                                             content  \
0  Sitharaman, the first full-time woman finance ...   
1  The merger of Tata group’s budget airlines Air...   
2  The Air India group plans to induct one aircra...   
3  Indian exporters have asked the central govern...   
4  The Air India group plans to induct one air

In [4]:
df = df.drop('category', axis=1)

In [5]:
def clean_text(text):
    if isinstance(text, str):
        text = text.lower()
        text = re.sub(r'\s+', ' ', text)
        text = re.sub(r'\W', ' ', text)
        text = re.sub(r'\d+', '', text)
        return text.strip()
    return ""

In [6]:
df['cleaned_headlines'] = df['headlines'].apply(clean_text)
df['cleaned_description'] = df['description'].apply(clean_text)
df['cleaned_content'] = df['content'].apply(clean_text)

In [7]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')  # Sometimes needed for WordNet Lemmatizer
nltk.download('averaged_perceptron_tagger')  # Needed for lemmatization
nltk.download('vader_lexicon')  # Needed for sentiment analysis

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


In [8]:
def preprocess_text(text):
    tokens = word_tokenize(text)  # Tokenization
    tokens = [word for word in tokens if word.isalpha()]
    tokens = [word for word in tokens if word not in stop_words]
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return " ".join(tokens)

df['processed_headlines'] = df['cleaned_headlines'].apply(preprocess_text)
df['processed_description'] = df['cleaned_description'].apply(preprocess_text)
df['processed_content'] = df['cleaned_content'].apply(preprocess_text)

# Combine all processed text into a single column
df['processed_text'] = df['processed_headlines'] + " " + df['processed_description'] + " " + df['processed_content']

In [11]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# Step 1: Define the number of categories (increased to 15 for better topic separation)
num_categories = 13

# Step 2: Vectorize the text data using TF-IDF
vectorizer = TfidfVectorizer(max_df=0.90, min_df=3, stop_words='english')
dtm = vectorizer.fit_transform(df['processed_text'])

# Step 3: Train the LDA model with improved settings
lda_model = LatentDirichletAllocation(n_components=num_categories, learning_method='online', random_state=42)
lda_model.fit(dtm)

# Step 4: Assign LDA topics to documents using confidence threshold
topic_distributions = lda_model.transform(dtm)
df['lda_topic'] = topic_distributions.argmax(axis=1)  # Assigning the topic with the highest probability
df['confidence'] = topic_distributions.max(axis=1)  # Extracting the confidence score

# Step 5: Define meaningful category names
category_mapping = {
    0: 'Politics',
    1: 'Business',
    2: 'Technology',
    3: 'Health',
    4: 'Sports',
    5: 'Entertainment',
    6: 'Science',
    7: 'World News',
    8: 'Economy',
    9: 'Environment',
    10: 'Education',
    11: 'Crime',
    12: 'Travel',
}

# Assign categories with confidence threshold
confidence_threshold = 0.3
df['predicted_category'] = df.apply(
    lambda row: category_mapping.get(row['lda_topic'], 'Uncertain') if row['confidence'] > confidence_threshold else 'Uncertain',
    axis=1
)

# Display Topics with the Predicted Category Labels
def display_topics_with_categories(model, feature_names, category_mapping, num_words=10):
    for idx, topic in enumerate(model.components_):
        category_label = category_mapping.get(idx, f"Unlabeled_Topic_{idx}")
        print(f"\nCategory: {category_label}")
        print("Keywords:", [feature_names[i] for i in topic.argsort()[-num_words:]])

display_topics_with_categories(lda_model, vectorizer.get_feature_names_out(), category_mapping)

# Step 6: Print a preview of the DataFrame with predicted categories
print(df[['processed_text', 'predicted_category', 'confidence']].head())


Category: Politics
Keywords: ['aibe', 'mht', 'mm', 'mohanlal', 'kangana', 'priyanka', 'cet', 'temple', 'mammootty', 'dhoni']

Category: Business
Keywords: ['quarter', 'index', 'sensex', 'point', 'salaar', 'market', 'bank', 'nifty', 'crore', 'cent']

Category: Technology
Keywords: ['lawcet', 'pgecet', 'ecet', 'ilt', 'tsche', 'icet', 'sehwag', 'apsche', 'eapcet', 'eamcet']

Category: Health
Keywords: ['kasautii', 'pande', 'virani', 'creditworthiness', 'utilization', 'finserv', 'nbfcs', 'advisable', 'udaan', 'cibil']

Category: Sports
Keywords: ['guarantee', 'renewable', 'wind', 'carbon', 'stamp', 'energy', 'mca', 'green', 'hydrogen', 'biofuels']

Category: Entertainment
Keywords: ['russian', 'lionel', 'nassr', 'liverpool', 'saudi', 'cristiano', 'madrid', 'djokovic', 'messi', 'ronaldo']

Category: Science
Keywords: ['ravikumar', 'bejoy', 'chittha', 'mumtaz', 'laal', 'extraterrestrial', 'seti', 'sivakarthikeyan', 'alien', 'ayalaan']

Category: World News
Keywords: ['lee', 'undone', 'kyun'

In [12]:
nltk.download('vader_lexicon')
sia = SentimentIntensityAnalyzer()

def get_sentiment(text):
    sentiment_score = sia.polarity_scores(text)['compound']
    return "Positive" if sentiment_score > 0.05 else "Negative" if sentiment_score < -0.05 else "Neutral"

df['sentiment'] = df['processed_text'].apply(get_sentiment)

# Display sentiment distribution
print(df['sentiment'].value_counts())

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


sentiment
Positive    8268
Negative    1445
Neutral      287
Name: count, dtype: int64


In [13]:
nlp = spacy.load("en_core_web_sm")

def extract_entities(text):
    doc = nlp(text)
    return [ent.text for ent in doc.ents if ent.label_ in ["ORG", "GPE", "PERSON"]]

df['named_entities'] = df['processed_text'].apply(extract_entities)

# Count top-mentioned entities
all_entities = [entity for sublist in df['named_entities'].dropna() for entity in sublist]
entity_counts = Counter(all_entities)

# Display top 10 entities
print(entity_counts.most_common(10))

[('india', 9590), ('google', 2281), ('australia', 1076), ('khan', 893), ('kapoor', 734), ('microsoft', 597), ('new zealand', 573), ('china', 532), ('russia', 408), ('singh', 407)]


In [14]:
df_subset = df.sample(n=100, random_state=42)

# SpaCy-based summarization function
def extract_summary_spacy(text, max_sentences=2):
    doc = nlp(text)
    sentences = [sent.text for sent in doc.sents]
    return " ".join(sentences[:max_sentences]) if len(sentences) > 1 else text

# Apply SpaCy summarization to the subset
df_subset['summary_spacy'] = df_subset['processed_text'].apply(lambda x: extract_summary_spacy(x))

In [15]:
from transformers import T5ForConditionalGeneration, T5Tokenizer, pipeline

# Load pre-trained T5 model and tokenizer
model_name = "t5-base"
model = T5ForConditionalGeneration.from_pretrained(model_name)
tokenizer = T5Tokenizer.from_pretrained(model_name)

# Initialize the summarizer pipeline
summarizer = pipeline("summarization", model=model, tokenizer=tokenizer)

# Define the summarization function using T5
def extract_summary_t5(text):
    if len(text.split()) < 10:
        return text
    summary = summarizer(text, max_length=10, min_length=2, do_sample=False)
    return summary[0]['summary_text']

# Apply T5 summarization to the 'processed_text' column
df_subset['summary_t5'] = df_subset['processed_text'].apply(lambda x: extract_summary_t5(x))

# Print the results
print(df_subset[['processed_text', 'summary_t5']].head())

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
Device set to use cuda:0
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


                                         processed_text  \
6252  mohammed shami likely miss test england suryak...   
4684  athiya shetty superstitious kl rahul playing c...   
1731  hindustan unilever profit rise june quarter ma...   
4742  vijayakanth tamil nadu loses captain actor pol...   
4521  aamir khan daughter ira khan reacts husband nu...   

                                      summary_t5  
6252                  mohammed shami likely miss  
4684                             kl rahul opened  
1731  hindustan's second largest service company  
4742                           vijayakanth tamil  
4521                          ira khan tied knot  


In [16]:
# Save the summarized data to a CSV file
df_subset.to_csv("summarized_news_subset_random.csv", index=False)

# Display the first few rows with summaries
print(df_subset[['processed_text', 'summary_t5']].head())

                                         processed_text  \
6252  mohammed shami likely miss test england suryak...   
4684  athiya shetty superstitious kl rahul playing c...   
1731  hindustan unilever profit rise june quarter ma...   
4742  vijayakanth tamil nadu loses captain actor pol...   
4521  aamir khan daughter ira khan reacts husband nu...   

                                      summary_t5  
6252                  mohammed shami likely miss  
4684                             kl rahul opened  
1731  hindustan's second largest service company  
4742                           vijayakanth tamil  
4521                          ira khan tied knot  


In [17]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 484.9/484.9 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 15.3 MB/s eta 0:00:00


In [30]:
from datasets import Dataset
from transformers import Trainer, TrainingArguments
from sklearn.model_selection import train_test_split

# Step 1: Prepare a dataset with text-summary pairs and limit to 250 data points
df_subset_limited = df_subset.head(250)  # Select the first 250 rows

data = {
    "text": df_subset_limited['processed_text'].tolist(),  # Your processed text (250 rows)
    "summary": df_subset_limited['summary_t5'].tolist()  # Your generated summaries (250 rows)
}

# Convert to Hugging Face Dataset
dataset = Dataset.from_dict(data)

# Step 2: Split the dataset into training and validation sets using Hugging Face Dataset's method
dataset = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = dataset['train']
val_dataset = dataset['test']

# Step 3: Preprocess the data (Tokenize the text and summaries)
def preprocess_function(examples):
    inputs = tokenizer(examples['text'], max_length=512, truncation=True, padding="max_length")
    outputs = tokenizer(examples['summary'], max_length=150, truncation=True, padding="max_length")
    inputs['labels'] = outputs['input_ids']
    return inputs

train_dataset = train_dataset.map(preprocess_function, batched=True)
val_dataset = val_dataset.map(preprocess_function, batched=True)

# Step 4: Initialize the Trainer for fine-tuning
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="no",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    save_steps=500,
    save_total_limit=1,
    report_to="none"
)

trainer = Trainer(
    model=model,  # Pre-trained T5 model
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
)

# Step 5: Start Fine-Tuning
trainer.train()

# Step 6: Save the fine-tuned model
model.save_pretrained('./fine_tuned_t5')
tokenizer.save_pretrained('./fine_tuned_t5')

# Step 7: Evaluate the model (optional)
results = trainer.evaluate()
print(results)

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-30-58fe11073eeb>:47: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
10,0.130200
20,0.094900
30,0.075100
40,0.057900
50,0.052400


{'eval_loss': 0.09049346297979355, 'eval_runtime': 1.1691, 'eval_samples_per_second': 17.107, 'eval_steps_per_second': 2.566, 'epoch': 5.0}


In [31]:
fine_tuned_model = T5ForConditionalGeneration.from_pretrained('./fine_tuned_t5')
fine_tuned_tokenizer = T5Tokenizer.from_pretrained('./fine_tuned_t5')

summarizer = pipeline("summarization", model=fine_tuned_model, tokenizer=fine_tuned_tokenizer)

Device set to use cuda:0


In [32]:
df_subset_limited['generated_summary'] = df_subset_limited['processed_text'].apply(
    lambda x: summarizer(x, max_length=10, min_length=2, do_sample=False)[0]['summary_text']
)
print(df_subset_limited[['processed_text', 'generated_summary']].head())

                                         processed_text  \
6252  mohammed shami likely miss test england suryak...   
4684  athiya shetty superstitious kl rahul playing c...   
1731  hindustan unilever profit rise june quarter ma...   
4742  vijayakanth tamil nadu loses captain actor pol...   
4521  aamir khan daughter ira khan reacts husband nu...   

               generated_summary  
6252  mohammed shami likely miss  
4684             kl rahul opened  
1731                 ltimindtree  
4742           vijayakanth tamil  
4521          ira khan tied knot  


<ipython-input-32-ba8a13813642>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_subset_limited['generated_summary'] = df_subset_limited['processed_text'].apply(
